In [1]:
import os
from pathlib import Path 
import pandas as pd

# 100 epochs

In [12]:
BASE_PATH = "/home/gkulemeyer/Documents/Repos/RNA-analysis/Experiments/f_ArchiveII/UFold/BRONZE/"

datasets = os.listdir(BASE_PATH)
datasets.sort()
del(datasets[0])  # skip lr
print(datasets[0])

fams = os.listdir(BASE_PATH+datasets[1])
fams.sort()
print(fams)

famfold
['16s', '23s', '5s', 'RNaseP', 'grp1', 'srp', 'tRNA', 'telomerase', 'tmRNA']


In [13]:
df = pd.read_csv(BASE_PATH+datasets[1]+"/"+fams[0]+"/"+"summary.csv")
display(df.head(2))
df.f1_exact.mean().item()

,name,p_appr1,s_appr1,f1_appr1,f1_exact,dataset,fold
0,16s_B.subtilis_domain4,0.652174,0.365854,0.468750,0.468750,hc_100,16s
1,16s_H.sapiens.mito_domain3,0.362319,0.294118,0.350649,0.324675,hc_100,16s


0.3904404572689076

In [4]:
# datasets = [datasets[0]]
datasets

['0_long_run100',
 'famfold',
 'hc_100',
 'hc_200',
 'hc_400',
 'rnadist_100',
 'rnadist_200',
 'rnadist_400',
 'samples_100',
 'samples_200',
 'samples_400']

In [18]:
tables = {d: {fam: 999 for fam in fams} for d in datasets}

In [6]:
for file in datasets:
    for fam in fams:
        data_path = BASE_PATH+file+"/"+fam+"/"+"summary.csv"
        if os.path.exists(data_path):
            df = pd.read_csv(data_path)
            tables[file][fam] = df.f1_exact.mean().item()
tables_df = pd.DataFrame(tables)

SAVE = True
if SAVE:
    SAVE_PATH = "SILVER/"
    os.makedirs(SAVE_PATH, exist_ok=True)
    tables_df.to_csv(f"{SAVE_PATH}/100_epochs_test_exact.csv")

In [7]:
for file in datasets:
    for fam in fams:
        data_path = BASE_PATH+file+"/"+fam+"/"+"summary.csv"
        if os.path.exists(data_path):
            df = pd.read_csv(data_path)
            tables[file][fam] = df.f1_appr1.mean().item()
tables_df = pd.DataFrame(tables)

SAVE = True
if SAVE:
    SAVE_PATH = "SILVER/"
    os.makedirs(SAVE_PATH, exist_ok=True)
    tables_df.to_csv(f"{SAVE_PATH}/100_epochs_test_appr1.csv")

# Long run

In [14]:
import shutil
BASE_PATH = "/home/gkulemeyer/Documents/Repos/RNA-analysis/Experiments/f_ArchiveII/UFold/"
BRONZE_PATH = os.path.join(BASE_PATH, "BRONZE/0_long_run100/")
SILVER_PATH = os.path.join(BASE_PATH, "SILVER/0_long_run100/")

epochs = [0, 4, 9, 14, 19, 24, 29, 34, 39, 44, 49, 54, 59, 64, 69, 74, 79, 84, 89, 94, 99]
metrics = ['p_appr1', 's_appr1', 'f1_appr1', 'f1_exact']

all_tables = {}
datasets = sorted(os.listdir(BRONZE_PATH))

if datasets:
    fams = sorted(os.listdir(os.path.join(BRONZE_PATH, datasets[0])))
else:
    fams = []

In [15]:
SAVE = True
for dset in datasets:
    for fam in fams:
        fam_data = []
        for ep in epochs:
            path = os.path.join(BRONZE_PATH, dset, fam)
            summary_path = os.path.join(path, f"{ep}", "summary.csv")

            if os.path.exists(summary_path):
                try:
                    df = pd.read_csv(summary_path)
                    if not df.empty and all(m in df.columns for m in metrics):
                        data = df[metrics].mean()
                        data['epoch'] = ep
                        fam_data.append(data)
                except pd.errors.EmptyDataError:
                    pass
        
        if fam_data:
            key = (dset, fam)
            all_tables[key] = pd.DataFrame(fam_data).sort_values(by='epoch')

            if SAVE:
                save_path = os.path.join(SILVER_PATH, dset, fam)
                os.makedirs(save_path, exist_ok=True)
                all_tables[key].to_csv(f"{save_path}/test_log.csv", index=False)
                
                shutil.copyfile(os.path.join(path, "train_log.csv"),
                                os.path.join(save_path, "train_log.csv"))


In [16]:
tables = {d: {fam: 999 for fam in fams} for d in datasets}

In [17]:
for file in datasets:
    for fam in fams:
        data_path = BRONZE_PATH+file+"/"+fam+"/99/"+"summary.csv"
        if os.path.exists(data_path):
            df = pd.read_csv(data_path)
            tables[file][fam] = df.f1_exact.mean().item()
tables_df = pd.DataFrame(tables)

SAVE = True
if SAVE:
    os.makedirs(SILVER_PATH, exist_ok=True)
    tables_df.to_csv(f"{SILVER_PATH}/100_epochs_lr_test_exact.csv")

In [18]:
for file in datasets:
    for fam in fams:
        data_path = BASE_PATH+file+"/"+fam+"/"+"summary.csv"
        if os.path.exists(data_path):
            df = pd.read_csv(data_path)
            tables[file][fam] = df.f1_appr1.mean().item()
tables_df = pd.DataFrame(tables)

SAVE = True
if SAVE:
    SAVE_PATH = "SILVER/"
    os.makedirs(SILVER_PATH, exist_ok=True)
    tables_df.to_csv(f"{SILVER_PATH}/100_epochs_test_appr1.csv")